In [31]:
'''
Clone github and pip install skorch if working in colab
'''

import os

try:
    import google.colab
    os.system('git clone https://github.com/ethanresek/luminal-classifiers 2>/dev/null; cd /content/luminal-classifiers && git pull')
    os.system('pip install skorch')
except ImportError:
    pass

In [44]:
import numpy as np
import pandas as pd
import sys
import joblib
import torch.nn.functional as F

from datetime import datetime
from typing import override
from torch import nn, ones, sigmoid, optim
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from skorch import NeuralNetBinaryClassifier
from skorch.callbacks import EarlyStopping
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score
print("Imports OK")

In [33]:
FEATURE_COUNTS = [5, 10, 20, 50]
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Imports OK


In [34]:
'''
Split set up instructions for local vs colab running
'''
try:
    from google.colab import drive
    sys.path.append('/content/luminal-classifiers')
    from pre_process import preprocess, split_data

    # Change CSV path if necessary
    # '/content/luminal-classifiers/' should remain the start of the path
    CSV = '/content/luminal-classifiers/data/METABRIC_RNA_Mutation.csv'
    print('Working in Colab')
except ImportError:
    from pre_process import preprocess, split_data

    # Change CSV to path in local storage if needed
    CSV = 'data/METABRIC_RNA_Mutation.csv'
    print('Working locally')

Working locally


In [35]:
DF = pd.read_csv(CSV, low_memory=False)

# Specify which columns to keep from dataframe
Y_OLD_NAME = 'pam50_+_claudin-low_subtype'
KEEP = list(DF.columns[31:520]) + [Y_OLD_NAME]

# Split data and convert to float32 for Torch
X, y = preprocess(DF, KEEP)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_SEED)
X_train, X_test, y_train, y_test = X_train.astype('float32').values, X_test.astype('float32').values, y_train.astype('float32').values, y_test.astype('float32').values

In [41]:
class SparsityNNBC(NeuralNetBinaryClassifier):
    '''
    Override NNBC model's get_loss model to apply sparsity effect to the loss.
    '''
    def __init__(self, **kwargs):
        super(SparsityNNBC, self).__init__(**kwargs)

    @override
    def get_loss(self, y_pred, y_true, X=None, training=False):
        sparsity_effect = self.module_.sparsity * sigmoid(self.module_.gate).sum()
        loss = super(SparsityNNBC, self).get_loss(y_pred, y_true, X, training) + sparsity_effect
        return loss

In [1]:
class MLP(nn.Module):
    '''
    Simple multi-layer perceptron model for classifying Luminal A/B breast cancer. Uses ReLu as an activation function for each layer.

    Inputs:
    sparsity - A penalty that is used to hit specific feature counts (relevant for feature selection in CS6140_MLP_Features)
    hidden_sizes - The size of each hidden layer
    dropout - probability that dropout is applied to elements


    Features:
    gate - A learnable parameter that can "shut off" specific features that are less important for cancer classification
    linear - Applies linear transformation to input
    batch normalize - normalize layer outputs to have zero mean and unit variance
    '''

    def __init__(self, hidden_sizes, dropout, sparsity, input_size=489, output_size=1):
        super(MLP, self).__init__()

        self.sparsity = sparsity
        self.gate = nn.Parameter(ones(input_size))

        sizes = [input_size] + hidden_sizes + [output_size]
        self.linears = nn.ModuleList([
            nn.Linear(sizes[i], sizes[i+1]) for i in range(len(sizes) - 1)
        ])
        self.dropout = nn.Dropout(dropout)
        self.batch_norms = nn.ModuleList([
            nn.BatchNorm1d(h) for h in hidden_sizes
        ])

    def forward(self, x):
        '''

        :param x: input data
        :return: output results
        '''

        x = x * sigmoid(self.gate)

        for i, l in enumerate(self.linears[:-1]):
            x = l(x)
            x = self.batch_norms[i](x)
            x = F.relu(x)
            x = self.dropout(x)

        x = self.linears[-1](x)

        return x

NameError: name 'nn' is not defined

In [43]:
# Set up DNN trainer with basic starter values so initial hyperparameter selection runs smoothly at beginning.
net = SparsityNNBC(
    module=MLP,
    criterion=nn.BCEWithLogitsLoss,
    optimizer=optim.Adam,
    max_epochs=100,
    callbacks=[EarlyStopping(patience=10)],
    module__hidden_sizes=[128, 64],
    module__dropout=0.3,
    module__input_size=489,
    module__sparsity=1e-3,
    optimizer__weight_decay=1e-3,
    verbose=0
)

# Distribution of possible values for hyperparameters
p_dist = {
    'module__hidden_sizes': [[256, 128], [128, 64], [64, 32], [128]],
    'module__dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
    'module__sparsity': [1e-4, 1e-3, 5e-3, 1e-2],
    'optimizer__weight_decay': [1e-4, 1e-3, 1e-2],
    'lr': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    'batch_size': [16, 32, 64],
}

In [ ]:
# Run randomized search to find optimal hyperparameters
rand_search = RandomizedSearchCV(net, param_distributions=p_dist, scoring='f1', n_iter=100, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), random_state=RANDOM_SEED, n_jobs=-1)
rand_search.fit(X_train, y_train)
print(rand_search.best_params_)

# For time purposes, GridSearch was not run on the Base model

In [46]:
# Produce F1, Balanced Accuracy, and ROC AUC scores

final_net = rand_search.best_estimator_

y_pred = final_net.predict(X_test)
y_pred_prob = final_net.predict_proba(X_test)[:, 1]

print('F1:', f1_score(y_test, y_pred))
print('Balanced Accuracy:', balanced_accuracy_score(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_pred_prob))


NameError: name 'rand_search' is not defined

In [45]:
# Store timestamped results of RandomizedSearch and input data sets
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs('models', exist_ok=True)

joblib.dump({
    'model': rand_search,
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test
}, f'models/tuned_MLP_search_{timestamp}.joblib')

NameError: name 'rand_search' is not defined

In [ ]:
'''
Use gates from base model for feature extraction
'''
gate = final_net.module_.gate
s_gate = sigmoid(gate).detach().numpy()
feature_names = X.columns
all_top_k_idx = []

for k in FEATURE_COUNTS:
    top_k_idx = np.argsort(s_gate)[-k:]
    all_top_k_idx.append(top_k_idx)
    top_k_names = feature_names[top_k_idx]
    print(f"Top {k}: {list(top_k_names)}")

In [ ]:
'''
Run randomized search on given number of features
'''
searches = []

for arr in all_top_k_idx:
    print("------------------------------------------------------")
    print(f"Searching for hyperparameters for {len(arr)} features")
    print("------------------------------------------------------")
    X_train_k = X_train[:, arr]
    X_test_k = X_test[:, arr]

    # Set up DNN trainer with basic starter values so initial hyperparameter selection runs smoothly at beginning.
    net = SparsityNNBC(
        module=MLP,
        criterion=nn.BCEWithLogitsLoss,
        optimizer=optim.Adam,
        max_epochs=100,
        callbacks=[EarlyStopping(patience=10)],
        module__hidden_sizes=[128, 64],
        module__dropout=0.3,
        module__input_size=len(arr),
        module__sparsity=1e-3,
        optimizer__weight_decay=1e-3,
        verbose=0
    )

    # Distribution of possible values for hyperparameters
    p_dist = {
        'module__hidden_sizes': [[256, 128], [128, 64], [64, 32], [128]],
        'module__dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
        'module__sparsity': [1e-4, 1e-3, 5e-3, 1e-2],
        'optimizer__weight_decay': [1e-4, 1e-3, 1e-2],
        'lr': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
        'batch_size': [16, 32, 64],
    }

    # Run randomized search to find range for hyperparameters
    rand_search = RandomizedSearchCV(net, param_distributions=p_dist, scoring='f1', n_iter=100, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), random_state=RANDOM_SEED, n_jobs=-1)
    rand_search.fit(X_train_k, y_train)

    y_pred = rand_search.predict(X_test_k)
    y_pred_prob = rand_search.predict_proba(X_test_k)[:, 1]

    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

    # Report accuracy scores for randomized search
    print("Accuracies for Randomized Search CV")
    print(f"F1-score: {f1}")
    print(f"ROC-AUC: {roc_auc}")
    print(f"Balanced Accuracy: {balanced_accuracy}")

    best = rand_search.best_params_

    # Create new ranges for possible hyperparameter values based on step sizes
    # Parameters with no natural step size are kept as is
    grid_params = {}
    for key, val in best.items():
        if (isinstance(val, str)
                or val is None
                or key == 'module__hidden_sizes'
                or key == 'batch_size'):
            grid_params[key] = [val]
        elif isinstance(val, int):
            step = max(1, round(0.1 * val))
            grid_params[key] = [max(1, val - step), val, val + step]
        elif isinstance(val, float):
            step = max(0.005, round(0.1 * val, 4))
            grid_params[key] = [max(0, round(val - step, 4)), val, val + step]

    post_rand_net = rand_search.best_estimator_

    # Run GridSearch for thorough examination of hyperparameters
    grid_search = GridSearchCV(post_rand_net, param_grid=grid_params, scoring='f1', cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), n_jobs=-1)
    grid_search.fit(X_train_k, y_train)

    # Produce F1, Balanced Accuracy, and ROC AUC scores
    y_pred = grid_search.predict(X_test_k)
    y_pred_prob = grid_search.predict_proba(X_test_k)[:, 1]

    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

    print("Accuracies for Grid Search CV")
    print(f"F1-score: {f1}")
    print(f"ROC-AUC: {roc_auc}")
    print(f"Balanced Accuracy: {balanced_accuracy}")

    searches.append(grid_search)

In [ ]:
# Save all searches as well as the indices used and the given feature counts
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs('models', exist_ok=True)
joblib.dump({
    'searches': searches,
    'feature_indices': all_top_k_idx,
    'feature_counts': FEATURE_COUNTS
}, f"models/MLP_grid_searches_{timestamp}.joblib")